# XR Engine V3: Distillation Foundry (Colab fallback)

This notebook runs the **foundry** side of V3 inverse rendering: take a
super-resolved frame plus G-buffers, recover albedo, and distill it into
a plain JSON parameter bundle. See `INVERSE.md` in the repo root.

**Two-sided design, the hard rule:** the foundry (this notebook, a local
GPU, or any DLSS capture setup) runs **once, offline**. Its output is a
plain JSON bundle evaluated by the engine's bit-exact TypeScript mirrors.
The **runtime** (the engine, the end user) has **zero** dependencies on
NVIDIA, Google Colab, DLSS, or any ML framework. Once the engine has the
math, no more DLSS is required.

**Which path are you?**

- **You have your own GPU computer:** that is the official path. Clone
  the repo and run the commands in the last section on your machine.
- **You do not:** this notebook is the fallback. It runs the same
  foundry on Colab's free GPU tier.

## 0. Environment check

Confirm we have a GPU (for real captures) and Python 3. The `xrgen`
package is **stdlib-only by design**, so there is nothing to pip install.

In [ ]:
import sys
print("python", sys.version)
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "no GPU visible (CPU-only run: synthetic demo still works)"

## 1. Get the engine source

Clone the repo (or upload it). The foundry lives in `python/xrgen/inverse.py`.

In [ ]:
!test -d xr-engine || git clone --depth 1 https://github.com/Frozen-In-Flames/xr-engine.git
import os
os.chdir("xr-engine/python")
print("cwd:", os.getcwd())
!python3 -c "import xrgen.inverse as m; print('inverse.py OK:', m.BUNDLE_FORMAT)"

## 2. Run the foundry on synthetic data

End-to-end on CPU: build a synthetic capture (known material rendered
with fake shading), splat it to texture space, strip the lighting,
fit the procedural detail, and emit a validated bundle.

With a real DLSS/upscaled frame, replace the synthetic capture with
`load_gbuffer_bundle()` on your capture directory (see section 4).

In [ ]:
import math, tempfile, json
from xrgen.inverse import (
    detail_reference, save_gbuffer_bundle, load_gbuffer_bundle,
    run_foundry, validate_bundle,
)

# Ground truth: the material this synthetic frame was 'rendered' from.
# In the real loop these params come from the engine material spec.
SRC = {"seed": [11.0, -23.0], "scale": 8.0, "octaves": 4}
W_TRUE = [0.45, 0.3, 0.15, 0.1, 0.0, 0.0]
BASE = (0.55, 0.35, 0.2)
W, H, TW, TH = 64, 64, 32, 32

frame, uv, depth, shading = [], [], [], []
for j in range(H):
    v = (j + 0.5) / H
    for i in range(W):
        u = (i + 0.5) / W
        d = detail_reference(u, v, SRC["seed"][0], SRC["seed"][1],
                             SRC["scale"], SRC["octaves"], W_TRUE, 1.0)
        s = 0.5 + 0.4 * u + 0.1 * v  # fake shading to be stripped
        for k in range(3):
            frame.append(BASE[k] * d * s)
            shading.append(s)
        uv += [u, v]
        depth.append(5.0 + 0.05 * math.sin(i * 1.7))

tmp = tempfile.mkdtemp()
save_gbuffer_bundle(f"{tmp}/cap", W, H, frame, uv, depth, shading)
cap = load_gbuffer_bundle(f"{tmp}/cap")
bundle = run_foundry(cap, "demo-terrain-mat", seed=20261010,
                     tex_w=TW, tex_h=TH, src=SRC,
                     provenance={"teacher": "synthetic"})
validate_bundle(bundle)
q = bundle["quality"]
print(f"R^2 score : {q['score']:.4f}")
print(f"rmse      : {q['rmse']:.6f}")
print(f"coverage  : {q['coverage']:.3f}")
print(f"passed    : {q['passed']}")
print(f"baseColor : {[round(c, 4) for c in bundle['material']['baseColor']]}")
print(f"weights   : {[round(x, 4) for x in bundle['material']['detail']['weights'][:4]]}")
print(f"true      : {W_TRUE[:4]} (up to the documented base/bias scale)")

## 3. Save the bundle

This JSON file is the **only** thing that crosses from the foundry to
the runtime. No weights files, no textures, no code. The demo loads it
with `loadDistillationBundle()` in `src/bridge/distill.ts`.

In [ ]:
with open("/tmp/distill-demo-terrain-mat.json", "w") as f:
    json.dump(bundle, f)
print("wrote /tmp/distill-demo-terrain-mat.json",
      f"({len(json.dumps(bundle))} bytes)")
print("quality.passed =", bundle["quality"]["passed"],
      "<- a failed fit is recorded, never silently shipped")

## 4. Real captures (your own GPU, or this Colab)

For a real asset:

1. **Render G-buffers** from the engine: open the First Light demo,
   call `renderer.captureGBuffer(root, camera, camNode, view)` from the
   console (foundry-only, flag-gated, needs `EXT_color_buffer_float`).
   Export the returned Float32Arrays with the manifest format in
   `save_gbuffer_bundle` (see `INVERSE.md` for the `.f32` layout).
2. **Super-resolve the frame** with DLSS, or any reconstruction tech,
   or a plain high-res render. Linear space in, linear space out;
   use `srgb_to_linear` from `xrgen.inverse` if your upscaler hands
   you sRGB.
3. **Run the foundry:**

```python
from xrgen.inverse import load_gbuffer_bundle, run_foundry, validate_bundle
cap = load_gbuffer_bundle("captures/terrain-mat")
bundle = run_foundry(
    cap, "terrain-mat", seed=20261010, tex_w=128, tex_h=128,
    src={"seed": [ox, oy], "scale": s, "octaves": o},  # from the material spec
    provenance={"teacher": "DLSS"},
)
validate_bundle(bundle)
```

4. **Use the bundle at runtime:** serve the JSON next to the demo and
   load it with `loadDistillationBundle()`; `applyDistillationBundle()`
   turns it into a `MaterialSpec` with the `distill` effect. The player
   never touches a GPU vendor SDK, Colab, or DLSS.

### Local-GPU commands (the official path)

```bash
git clone https://github.com/Frozen-In-Flames/xr-engine.git
cd xr-engine/python
# xrgen is stdlib-only: no pip install needed.
python3 -m unittest discover -s test -p "test_*.py"   # foundry tests
python3 - <<'EOF'
from xrgen.inverse import load_gbuffer_bundle, run_foundry, validate_bundle
cap = load_gbuffer_bundle("captures/terrain-mat")
bundle = run_foundry(cap, "terrain-mat", seed=20261010,
                     tex_w=128, tex_h=128,
                     src={"seed": [11.0, -23.0], "scale": 8.0, "octaves": 4},
                     provenance={"teacher": "DLSS"})
validate_bundle(bundle)
print("score:", bundle["quality"]["score"],
      "passed:", bundle["quality"]["passed"])
EOF
```